In [20]:
import pandas as pd
import os
import glob

# Set the path to the directory containing the CSV files
path = r'c:\Users\Aidam\OneDrive\Desktop\Kuliah\DSA - Super() Bubur\Graduation Night\data combine'
all_files = glob.glob(os.path.join(path, "*.csv"))

df_list = []
file_prefix = 'Tingkat Pengangguran Terbuka (TPT) dan Tingkat Partisipasi Angkatan Kerja (TPAK) Menurut Kabupaten_Kota di Provinsi '

for filename in all_files:
    # Check if the filename contains the expected prefix
    if file_prefix in os.path.basename(filename):
        try:
            # Read the CSV file
            df = pd.read_csv(filename, index_col=None, header=0)
            
            # Extract province name from the filename
            base_name = os.path.basename(filename)
            province_part = base_name.replace(file_prefix, '')
            province_name = province_part.split(',')[0]
            
            # Add a 'Provinsi' column
            df['Provinsi'] = province_name
            
            df_list.append(df)
        except Exception as e:
            print(f"Could not process file {filename}: {e}")

# Concatenate all dataframes in the list
if df_list:
    combined_df = pd.concat(df_list, axis=0, ignore_index=True)
    print("Combined DataFrame created successfully.")
    # Display the first few rows of the combined dataframe
    print(combined_df.head())
else:
    print("No dataframes were created. Please check the file paths and format.")

Combined DataFrame created successfully.
  Kabupaten/Kota Tingkat Pengangguran Terbuka (TPT) - Februari  \
0       Simeulue                                           ...   
1   Aceh Singkil                                           ...   
2   Aceh Selatan                                           ...   
3  Aceh Tenggara                                           ...   
4     Aceh Timur                                           ...   

   Tingkat Pengangguran Terbuka (TPT) - Agustus  \
0                                          5.85   
1                                          6.84   
2                                          4.73   
3                                          5.00   
4                                          8.03   

  Tingkat Partisipasi Angkatan Kerja (TPAK) - Februari  \
0                                                ...     
1                                                ...     
2                                                ...     
3                      

In [21]:
combined_df

,Kabupaten/Kota,Tingkat Pengangguran Terbuka (TPT) - Februari,Tingkat Pengangguran Terbuka (TPT) - Agustus,Tingkat Partisipasi Angkatan Kerja (TPAK) - Februari,Tingkat Partisipasi Angkatan Kerja (TPAK) - Agustus,Provinsi
0,Simeulue,...,5.85,...,70.57,Aceh
1,Aceh Singkil,...,6.84,...,57.81,Aceh
2,Aceh Selatan,...,4.73,...,58.87,Aceh
3,Aceh Tenggara,...,5.00,...,70.43,Aceh
4,Aceh Timur,...,8.03,...,60.48,Aceh
...,...,...,...,...,...,...
598,Kota Gunungsitoli,...,3.67,...,67.77,Sumatera Utara
599,Sumatera Utara,5.24,5.89,70.6,71.06,Sumatera Utara
600,NaN,NaN,NaN,NaN,NaN,Sumatera Utara
601,Catatan,NaN,NaN,NaN,NaN,Sumatera Utara


In [22]:
# Drop the 'Februari' columns
cleaned_df = combined_df.drop(columns=[
    'Tingkat Pengangguran Terbuka (TPT) - Februari',
    'Tingkat Partisipasi Angkatan Kerja (TPAK) - Februari'
])

# Remove rows with NaN values in 'Kabupaten/Kota'
cleaned_df.dropna(subset=['Kabupaten/Kota'], inplace=True)

# Remove metadata rows by filtering based on 'Kabupaten/Kota' content
cleaned_df = cleaned_df[~cleaned_df['Kabupaten/Kota'].str.contains('Catatan|Kesalahan Baku', na=False)]

# Rename the 'Agustus' columns for simplicity
cleaned_df = cleaned_df.rename(columns={
    'Tingkat Pengangguran Terbuka (TPT) - Agustus': 'TPT Agustus',
    'Tingkat Partisipasi Angkatan Kerja (TPAK) - Agustus': 'TPAK Agustus'
})

# Reset the index of the cleaned dataframe
cleaned_df.reset_index(drop=True, inplace=True)

# Display the first few rows and info of the cleaned dataframe
print("Cleaned DataFrame:")
cleaned_df

Cleaned DataFrame:


,Kabupaten/Kota,TPT Agustus,TPAK Agustus,Provinsi
0,Simeulue,5.85,70.57,Aceh
1,Aceh Singkil,6.84,57.81,Aceh
2,Aceh Selatan,4.73,58.87,Aceh
3,Aceh Tenggara,5.00,70.43,Aceh
4,Aceh Timur,8.03,60.48,Aceh
...,...,...,...,...
502,Kota Medan,8.67,64.67,Sumatera Utara
503,Kota Binjai,6.10,62.79,Sumatera Utara
504,Kota Padangsidimpuan,7.57,68.90,Sumatera Utara
505,Kota Gunungsitoli,3.67,67.77,Sumatera Utara


In [23]:
# Rename columns for better readability
cleaned_df = cleaned_df.rename(columns={
    'TPT Agustus': 'Tingkat Pengangguran Terbuka',
    'TPAK Agustus': 'Tingkat Partisipasi Angkatan Kerja'
})

# Display the first few rows of the dataframe with new column names
cleaned_df

,Kabupaten/Kota,Tingkat Pengangguran Terbuka,Tingkat Partisipasi Angkatan Kerja,Provinsi
0,Simeulue,5.85,70.57,Aceh
1,Aceh Singkil,6.84,57.81,Aceh
2,Aceh Selatan,4.73,58.87,Aceh
3,Aceh Tenggara,5.00,70.43,Aceh
4,Aceh Timur,8.03,60.48,Aceh
...,...,...,...,...
502,Kota Medan,8.67,64.67,Sumatera Utara
503,Kota Binjai,6.10,62.79,Sumatera Utara
504,Kota Padangsidimpuan,7.57,68.90,Sumatera Utara
505,Kota Gunungsitoli,3.67,67.77,Sumatera Utara


In [24]:
# Get a list of all column names
cols = cleaned_df.columns.tolist()

# Move the 'Provinsi' column to the first position
cols.insert(0, cols.pop(cols.index('Provinsi')))

# Reorder the dataframe with the new column order
cleaned_df = cleaned_df[cols]

# Display the first few rows of the reordered dataframe
cleaned_df

,Provinsi,Kabupaten/Kota,Tingkat Pengangguran Terbuka,Tingkat Partisipasi Angkatan Kerja
0,Aceh,Simeulue,5.85,70.57
1,Aceh,Aceh Singkil,6.84,57.81
2,Aceh,Aceh Selatan,4.73,58.87
3,Aceh,Aceh Tenggara,5.00,70.43
4,Aceh,Aceh Timur,8.03,60.48
...,...,...,...,...
502,Sumatera Utara,Kota Medan,8.67,64.67
503,Sumatera Utara,Kota Binjai,6.10,62.79
504,Sumatera Utara,Kota Padangsidimpuan,7.57,68.90
505,Sumatera Utara,Kota Gunungsitoli,3.67,67.77


In [28]:
# Define the custom order for provinces
province_order = [
    'Aceh', 'Sumatera Utara', 'Sumatera Barat', 'Riau', 'Jambi', 'Sumatera Selatan',
    'Bengkulu', 'Lampung', 'Kepulauan Bangka Belitung', 'Kepulauan Riau', 'DKI Jakarta',
    'Jawa Barat', 'Jawa Tengah', 'DI Yogyakarta', 'Jawa Timur', 'Banten', 'Bali',
    'Nusa Tenggara Barat', 'Nusa Tenggara Timur', 'Kalimantan Barat', 'Kalimantan Tengah',
    'Kalimantan Selatan', 'Kalimantan Timur', 'Kalimantan Utara', 'Sulawesi Utara',
    'Sulawesi Tengah', 'Sulawesi Selatan', 'Sulawesi Tenggara', 'Gorontalo',
    'Sulawesi Barat', 'Maluku', 'Maluku Utara', 'Papua Barat', 'Papua'
]

# Convert the 'Provinsi' column to a categorical type with the specified order
cleaned_df['Provinsi'] = pd.Categorical(cleaned_df['Provinsi'], categories=province_order, ordered=True)

# Sort the dataframe by the 'Provinsi```python
# Define the custom order for provinces
province_order = [
    'Aceh', 'Sumatera Utara', 'Sumatera Barat', 'Riau', 'Jambi', 'Sumatera Selatan',
    'Bengkulu', 'Lampung', 'Kepulauan Bangka Belitung', 'Kepulauan Riau', 'DKI Jakarta',
    'Jawa Barat', 'Jawa Tengah', 'DI Yogyakarta', 'Jawa Timur', 'Banten', 'Bali',
    'Nusa Tenggara Barat', 'Nusa Tenggara Timur', 'Kalimantan Barat', 'Kalimantan Tengah',
    'Kalimantan Selatan', 'Kalimantan Timur', 'Kalimantan Utara', 'Sulawesi Utara',
    'Sulawesi Tengah', 'Sulawesi Selatan', 'Sulawesi Tenggara', 'Gorontalo',
    'Sulawesi Barat', 'Maluku', 'Maluku Utara', 'Papua Barat', 'Papua'
]

# Convert the 'Provinsi' column to a categorical type with the specified order
cleaned_df['Provinsi'] = pd.Categorical(cleaned_df['Provinsi'], categories=province_order, ordered=True)

# Sort the dataframe by the 'Provinsi
cleaned_df = cleaned_df.sort_values('Provinsi')

# Reset the index after sorting
cleaned_df.reset_index(drop=True, inplace=True)

# Display the sorted dataframe
cleaned_df

,Provinsi,Kabupaten/Kota,Tingkat Pengangguran Terbuka,Tingkat Partisipasi Angkatan Kerja
0,Aceh,Simeulue,5.85,70.57
1,Aceh,Aceh,6.03,64.77
2,Aceh,Kota Subulussalam,5.69,67.75
3,Aceh,Kota Lhokseumawe,8.78,64.36
4,Aceh,Kota Langsa,7.73,59.98
...,...,...,...,...
502,Papua,Biak Numfor,5.72,65.97
503,Papua,Kepulauan Yapen,4.01,61.05
504,Papua,Jayapura,4.07,67.69
505,Papua,Kota Jayapura,10.76,67.37


In [29]:
# Save the cleaned dataframe to a new CSV file
cleaned_df.to_csv('Klasifikasi Pengangguran Indonesia 2023.csv', index=False)

print("DataFrame has been saved to 'Klasifikasi Pengangguran Indonesia 2023.csv'")

DataFrame has been saved to 'Klasifikasi Pengangguran Indonesia 2023.csv'
